In [11]:
import textwrap
import os
import re
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

def _slug(texto: str) -> str:
    """
    Convierte texto a nombre de archivo seguro y estable.
    """
    texto = str(texto).strip().lower()
    texto = re.sub(r"\s+", "_", texto)
    texto = re.sub(r"[^a-z0-9_\-\.]+", "", texto)
    return texto[:180] if len(texto) > 180 else texto


import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.lines import Line2D


def graficar_sg_sv_por_clase(
    resumen: pd.DataFrame,
    dataset: str,
    configuracion: str,
    directorio_graficos: str = "graficos",
    mostrar_en_notebook: bool = True,
    guardar: bool = True,
    dpi: int = 140,
) -> None:
    """
    Genera:
      1) Burbuja SV vs SG (tamaño ~ SG/SV) con leyenda de marcadores fijos
      2) Barra SG/SV por clase

    Guarda en /graficos con nombres determinísticos por dataset+config,
    por lo que se SOBREESCRIBEN al re-ejecutar.

    Espera 'resumen' con índice=clase_objetivo y columnas: SG, SV.
    Requiere que exista la función _slug(texto: str) -> str.
    """

    # -------------------------
    # Validaciones mínimas
    # -------------------------
    if resumen is None or len(resumen) == 0:
        print("  ▶ Gráficos: resumen vacío (omitido)")
        return

    if "SG" not in resumen.columns or "SV" not in resumen.columns:
        print("  ▶ Gráficos: faltan columnas SG/SV en resumen (omitido)")
        return

    # -------------------------
    # Preparación de datos
    # -------------------------
    clases = []
    lista_sv = []
    lista_sg = []
    lista_ratio = []

    for clase, fila in resumen.iterrows():
        sg = float(fila["SG"]) if pd.notna(fila["SG"]) else 0.0
        sv = float(fila["SV"]) if pd.notna(fila["SV"]) else 0.0
        ratio = (sg / sv) if sv > 0 else 0.0

        clases.append(str(clase))
        lista_sv.append(sv)
        lista_sg.append(sg)
        lista_ratio.append(ratio)

    # Escala de tamaño (área en puntos^2)
    max_ratio = max(lista_ratio) if len(lista_ratio) > 0 else 0.0
    tamanios = []
    for r in lista_ratio:
        if max_ratio > 0:
            tamanio = 80.0 + (r / max_ratio) * 1320.0
        else:
            tamanio = 80.0
        tamanios.append(tamanio)

    # -------------------------
    # Rutas determinísticas
    # -------------------------
    base = f"{_slug(dataset)}__{_slug(configuracion)}"
    out_dir = Path(directorio_graficos)
    if guardar:
        out_dir.mkdir(parents=True, exist_ok=True)

    ruta_burbuja = out_dir / f"{base}__burbuja_sv_vs_sg.png"
    ruta_barra = out_dir / f"{base}__barra_sg_sv.png"

    # ============================================================
    # 1) Burbuja: SV vs SG (tamaño ~ SG/SV) + leyenda fija
    # ============================================================
    fig1 = plt.figure()
    ax = plt.gca()

    # Graficar SOLO activos y capturar color real por clase
    color_por_clase = {}
    clases_inactivas = []

    for i in range(len(clases)):
        sv = lista_sv[i]
        sg = lista_sg[i]

        if sv > 0 and sg > 0:
            sc = plt.scatter([sv], [sg], s=[tamanios[i]], alpha=0.7)

            face = sc.get_facecolors()
            if face is not None and len(face) > 0:
                color_por_clase[clases[i]] = face[0]
            else:
                color_por_clase[clases[i]] = (0.2, 0.2, 0.2, 1.0)
        else:
            clases_inactivas.append(clases[i])

    plt.title(f"{dataset} \n {configuracion} \n Burbuja SV vs SG (tamaño ~ SG/SV)")
    plt.xlabel("SV (semillas válidas)")
    plt.ylabel("SG (sintéticos generados)")
    plt.grid(True, which="both", linestyle="--", linewidth=0.5, alpha=0.5)
    # ---- forzar ticks en eje X para cada SV observado ----
    sv_unicos = sorted({int(sv) for sv in lista_sv if sv > 0})

    plt.xticks(sv_unicos)
    # ---- forzar ticks en eje Y para cada SG observado ----
    sg_unicos = sorted({int(sg) for sg in lista_sg if sg > 0})

    plt.yticks(sg_unicos)

    plt.margins(x=0.12, y=0.15)

    # ---------- LEYENDA PRINCIPAL ----------
    handles = []
    labels = []

    for nombre_clase in color_por_clase.keys():
        handles.append(
            Line2D(
                [0], [0],
                marker="o",
                linestyle="None",
                markerfacecolor=color_por_clase[nombre_clase],
                markeredgecolor="none",
                markersize=9,
                alpha=0.85
            )
        )
        labels.append(nombre_clase)

    leg1 = ax.legend(
        handles, labels,
        loc="upper left",
        bbox_to_anchor=(1.02, 1.0),
        borderaxespad=0.0,
        fontsize=9,
        title="Clase"
    )

    # ---------- LEYENDA SECUNDARIA (SIN GENERACIÓN) ----------
    if len(clases_inactivas) > 0:

        handle_inactivo = Line2D(
            [0], [0],
            marker="o",
            linestyle="None",
            markerfacecolor="lightgray",
            markeredgecolor="none",
            markersize=9,
            alpha=0.8
        )

        texto_inact = (
            "Sin generación (SV=0 o SG=0):\n"
            + "\n".join(f"- {c}" for c in clases_inactivas)
        )


        # MUY IMPORTANTE: re-agregar la leyenda principal
        ax.add_artist(leg1)

        ax.legend(
            [handle_inactivo],
            [texto_inact],
            loc="upper left",
            bbox_to_anchor=(1.02, 0.55),
            borderaxespad=0.0,
            fontsize=9,
            frameon=False
        )


    if guardar:
        fig1.savefig(ruta_burbuja, dpi=dpi, bbox_inches="tight")  # sobreescribe

    if mostrar_en_notebook:
        plt.show()

    plt.close(fig1)


    # ============================================================
    # 2) Dumbbell / Lollipop: SV vs SG por clase (escala log)
    # ============================================================
    # Filtrar activos e inactivos
    clases_act = []
    sv_act = []
    sg_act = []

    clases_inact = []
    for i in range(len(clases)):
        if lista_sv[i] > 0 and lista_sg[i] > 0:
            clases_act.append(clases[i])
            sv_act.append(float(lista_sv[i]))
            sg_act.append(float(lista_sg[i]))
        else:
            clases_inact.append(clases[i])

    # Ordenar por SG/SV (desc) para lectura
    indices = list(range(len(clases_act)))
    indices.sort(key=lambda k: (sg_act[k] / sv_act[k]) if sv_act[k] > 0 else 0.0, reverse=True)

    clases_ord = []
    sv_ord = []
    sg_ord = []
    ratio_ord = []

    for k in indices:
        clases_ord.append(clases_act[k])
        sv_ord.append(sv_act[k])
        sg_ord.append(sg_act[k])
        ratio_ord.append(sg_act[k] / sv_act[k])

    fig2 = plt.figure()

    y_pos = list(range(len(clases_ord)))

    # línea entre SV y SG (con dirección semántica correcta)
    for j in range(len(clases_ord)):
        sv = sv_ord[j]
        sg = sg_ord[j]

        if sg >= sv:
            x_ini = sv
            x_fin = sg
        else:
            x_ini = sg
            x_fin = sv

        plt.plot([x_ini, x_fin], [y_pos[j], y_pos[j]], linewidth=2, alpha=0.7)


    label_sv = textwrap.fill("SV (semillas válidas)", width=16)
    label_sg = textwrap.fill("SG (sintéticos generados)", width=16)

    # puntos SV y SG
    plt.scatter(sv_ord, y_pos, s=60, alpha=0.9, label=label_sv)
    plt.scatter(sg_ord, y_pos, s=60, alpha=0.9, label=label_sg)

    ax = plt.gca()  # asegurate de tener el eje

    for j in range(len(clases_ord)):
        sv = sv_ord[j]
        sg = sg_ord[j]

        if sg >= sv:
            x_ini = sv
            x_fin = sg
            texto = f"SG = {sg / sv:.1f} veces SV"
        else:
            x_ini = sg
            x_fin = sv
            texto = f"SG = {sg / sv:.1f} × SV \n (sub - genero menos sinteticos)"

        # centro visual correcto en escala log
        x_centro = (x_ini * x_fin) ** 0.5

        if j == len(clases_ord) - 1:
            dy = +10
            va = "bottom"
        else:
            dy = -10
            va = "top"

        ax.annotate(
            texto,
            xy=(x_centro, y_pos[j]),
            xytext=(0, dy),
            textcoords="offset points",
            ha="center",
            va=va,
            fontsize=9
        )

    plt.yticks(y_pos, clases_ord)
    plt.gca().invert_yaxis()

    plt.title(f"{dataset} \n {configuracion} \n Relación SV vs SG por clase \n (dumbbell, escala log[10](SG/SV))")
    plt.xlabel("Cantidad (escala log10)")
    plt.ylabel("Clase")
    plt.grid(True, axis="x", linestyle="--", linewidth=0.5, alpha=0.5)

    # clave: escala log para que SV no quede pegado al cero
    plt.xscale("log")

    # ---- leyenda fuera del área del gráfico (a la derecha) ----
    plt.legend(
        loc="upper left",
        bbox_to_anchor=(1.02, 1.0),
        borderaxespad=0.0,
        fontsize=9
    )

    plt.tight_layout()

    if guardar:
        ruta_relacion = out_dir / f"{base}__dumbbell_sv_vs_sg.png"
        fig2.savefig(ruta_relacion, dpi=dpi, bbox_inches="tight")

    if mostrar_en_notebook:
        plt.show()

    plt.close(fig2)

    # Nota informativa si hay inactivos
    if len(clases_inact) > 0:
        print(f"  ▶ Dumbbell: clases sin generación (SV=0 o SG=0): {', '.join(clases_inact)}")



In [12]:

# =========================
# UTILIDADES
# =========================
def tiene_columnas(df: pd.DataFrame, columnas: list) -> bool:
    for c in columnas:
        if c not in df.columns:
            return False
    return True


def validar_columnas_obligatorias(df: pd.DataFrame, nombre_archivo: str) -> None:
    columnas_obligatorias = [
        "configuracion",
        "clase_objetivo",
        "synthetics_from_this_seed",
        "es_semilla_valida",
    ]

    faltantes = []
    for c in columnas_obligatorias:
        if c not in df.columns:
            faltantes.append(c)

    if len(faltantes) > 0:
        raise ValueError(f"Columnas faltantes en {nombre_archivo}: {faltantes}")


def convertir_a_float_seguro(serie: pd.Series) -> pd.Series:
    """
    Convierte una serie a float de forma robusta:
    - numéricos -> float
    - strings "7/7" -> 1.0
    - strings con coma decimal -> reemplaza coma por punto
    - cualquier otra cosa no convertible -> NaN
    """
    # Paso 1: forzamos a string para procesar patrones, pero preservamos NaN
    s = serie.copy()

    # Normalizamos a string sin perder NaN
    s = s.astype("object")

    valores_convertidos = []

    for v in s.values:
        if pd.isna(v):
            valores_convertidos.append(float("nan"))
            continue

        # Si ya es numérico
        if isinstance(v, (int, float)):
            valores_convertidos.append(float(v))
            continue

        texto = str(v).strip()

        if texto == "":
            valores_convertidos.append(float("nan"))
            continue

        # Caso "a/b"
        if "/" in texto:
            partes = texto.split("/")
            if len(partes) == 2:
                a_txt = partes[0].strip()
                b_txt = partes[1].strip()
                try:
                    a = float(a_txt.replace(",", "."))
                    b = float(b_txt.replace(",", "."))
                    if b != 0:
                        valores_convertidos.append(a / b)
                    else:
                        valores_convertidos.append(float("nan"))
                except Exception:
                    valores_convertidos.append(float("nan"))
                continue

        # Caso decimal con coma
        texto = texto.replace(",", ".")

        try:
            valores_convertidos.append(float(texto))
        except Exception:
            valores_convertidos.append(float("nan"))

    return pd.Series(valores_convertidos, index=serie.index)


In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.patches import Patch

def graficar_dot_box_por_criterio(
    bloque: pd.DataFrame,
    dataset: str,
    configuracion: str,
    directorio_graficos: str = "graficos",
    guardar: bool = True,
    mostrar_en_notebook: bool = True,
    dpi: int = 140,
    col_clase: str = "clase_objetivo",
) -> None:
    
    if bloque is None or bloque.empty:
        return

    # Al inicio de la función, después de validar que el bloque no esté vacío:
    config_real = bloque['configuracion'].iloc[0] if 'configuracion' in bloque.columns else configuracion


    # 1) Mapeo de columnas técnicas
    col_valor_pureza = "proporcion_min_valor" if "proporcion_min_valor" in bloque.columns else "entropia"
    col_densidad = "densidad_valor" if "densidad_valor" in bloque.columns else "densidad"
    col_riesgo = "riesgo_valor" if "riesgo_valor" in bloque.columns else "riesgo"
    
    col_pasa_pureza = "pasa_pureza"
    col_pasa_densidad = "pasa_densidad" if "pasa_densidad" in bloque.columns else "pasa_densida_d"
    col_pasa_riesgo = "pasa_riesgo"

    criterios = [
        {"nombre": "Pureza",   "col_pasa": col_pasa_pureza,   "col_valor": col_valor_pureza},
        {"nombre": "Densidad", "col_pasa": col_pasa_densidad, "col_valor": col_densidad},
        {"nombre": "Riesgo",   "col_pasa": col_pasa_riesgo,   "col_valor": col_riesgo},
    ]

    out_dir = Path(directorio_graficos)
    if guardar: out_dir.mkdir(parents=True, exist_ok=True)

    col_no = "#1f77b4" # Azul
    col_si = "#ff7f0e" # Naranja

    for crit in criterios:
        nombre_eje = crit["nombre"]
        c_pasa = crit["col_pasa"]
        c_valor = crit["col_valor"]

        if c_pasa not in bloque.columns or c_valor not in bloque.columns:
            continue

        clases_unicas = sorted(bloque[col_clase].unique())
        
        datos_plot = []
        posiciones = []
        etiquetas = []
        colores = []
        info_muestral = []
        
        curr_pos = 0.0
        for clase in clases_unicas:
            sub = bloque[bloque[col_clase] == clase].copy()
            sub['pasa_bool'] = sub[c_pasa].astype(str).str.upper().isin(['VERDADERO', 'TRUE', '1', 'SI', 'SÍ'])
            
            n_pasa = 0
            n_no_pasa = 0
            
            for estado, texto, color in [(False, "no pasa", col_no), (True, "pasa", col_si)]:
                vals = pd.to_numeric(sub[sub['pasa_bool'] == estado][c_valor], errors='coerce').dropna().values
                if len(vals) > 0:
                    datos_plot.append(vals)
                    posiciones.append(curr_pos)
                    etiquetas.append(f"Clase {clase}\n{texto}")
                    colores.append(color)
                    if estado: n_pasa = len(vals)
                    else: n_no_pasa = len(vals)
                    curr_pos += 1.0
            
            info_muestral.append(f"• Clase {clase}: no pasa={n_no_pasa}, pasa={n_pasa}")
            curr_pos += 0.8 

        if not datos_plot: continue

        # Aumentamos el margen derecho para que entren las leyendas alineadas
        fig, ax = plt.subplots(figsize=(15, 7))
        
        # 1. Boxplot (con transparencia)
        bp = ax.boxplot(
            datos_plot, 
            positions=posiciones, 
            widths=0.6, 
            patch_artist=True, 
            showfliers=False,
            medianprops=dict(color="black", linewidth=2.0),
            zorder=2
        )
        
        for patch, color in zip(bp['boxes'], colores):
            patch.set_facecolor(color)
            patch.set_alpha(0.2) # Transparente para que se vean los puntos
            patch.set_edgecolor(color)

        # 2. Re-introducir Puntos (Scatter con Jitter)
        for i, vals in enumerate(datos_plot):
            x_jitter = np.random.normal(posiciones[i], 0.04, size=len(vals))
            ax.scatter(x_jitter, vals, color=colores[i], s=25, alpha=0.6, zorder=3)

        # Configuración de Ejes
        ax.set_xticks(posiciones)
        ax.set_xticklabels(etiquetas, fontsize=9, rotation=45, ha='right')
        ax.set_ylabel(nombre_eje, fontsize=11, fontweight='bold')
        ax.set_title(f"{dataset} - Criterio {nombre_eje} por clase\nConfig: {config_real}", fontsize=12, fontweight='bold')
        ax.set_ylim(-0.05, 1.05)
        ax.grid(True, axis='y', linestyle='--', alpha=0.3)

        # 3. LEYENDAS ALINEADAS EN COLUMNA
        # Leyenda de resultado (Cajas)
        legend_elements = [
            Patch(facecolor=col_no, alpha=0.4, label='No Pasa'),
            Patch(facecolor=col_si, alpha=0.4, label='Pasa')
        ]
        
        leg1 = ax.legend(handles=legend_elements, title="Resultado", 
                         loc='upper left', bbox_to_anchor=(1.02, 1.0), frameon=True)
        
        # Texto de tamaño muestral (debajo de la leyenda anterior)
        texto_n = "Tamaño Muestral:\n" + "\n".join(info_muestral)
        ax.text(1.02, 0.75, texto_n, transform=ax.transAxes, verticalalignment='top',
                fontsize=9, bbox=dict(boxstyle='round', facecolor='white', alpha=0.5, edgecolor='gray'))

        plt.tight_layout(rect=[0, 0, 0.85, 1]) # Ajusta el área del gráfico para dejar espacio a la derecha

        if guardar:
            fn = f"{dataset}_dotbox_{nombre_eje}_{configuracion}.png".replace("/", "_")
            fig.savefig(out_dir / fn, dpi=dpi, bbox_inches='tight')
        if mostrar_en_notebook:
            plt.show()
        plt.close(fig)

In [14]:
import numpy as np
import pandas as pd


def _gini_no_negativos(valores: np.ndarray) -> float:
    """
    Gini para vector >=0.
    0 = perfectamente distribuido, 1 = totalmente concentrado.
    """
    x = np.asarray(valores, dtype=float)
    x = x[np.isfinite(x)]
    x = x[x >= 0]

    if x.size == 0:
        return np.nan

    suma = float(x.sum())
    if suma <= 0:
        return 0.0  # si todos son 0, no hay concentración (lo tratamos como 0)

    x = np.sort(x)
    n = x.size
    idx = np.arange(1, n + 1, dtype=float)

    # fórmula estándar
    g = (2.0 * np.sum(idx * x)) / (n * suma) - (n + 1.0) / n
    return float(np.clip(g, 0.0, 1.0))


def _equidad_generacion_desde_bloque(bloque: pd.DataFrame) -> float:
    """
    1 - Gini sobre SG_por_semilla para SV=1.
    1 = generación bien repartida.
    0 = muy concentrada en pocas semillas.
    """
    if "es_semilla_valida" not in bloque.columns or "synthetics_from_this_seed" not in bloque.columns:
        return np.nan

    sv = bloque[bloque["es_semilla_valida"] == 1]
    if len(sv) == 0:
        return np.nan

    sg = pd.to_numeric(sv["synthetics_from_this_seed"], errors="coerce").fillna(0).to_numpy()
    g = _gini_no_negativos(sg)
    if pd.isna(g):
        return np.nan

    return float(1.0 - g)  # alto = mejor


def _normalizar_flag_a_bool(serie: pd.Series) -> pd.Series:
    """
    Normaliza flags tipo True/False, 1/0, 'VERDADERO'/'FALSO', 'SI'/'NO'.
    Devuelve una serie booleana (True/False). Los valores no reconocidos -> False.
    """
    if serie is None:
        return pd.Series([], dtype=bool)

    if serie.dtype == bool:
        return serie.fillna(False)

    s = serie.astype(str).str.strip().str.upper()

    verdaderos = {"VERDADERO", "TRUE", "1", "SI", "SÍ", "YES", "Y"}
    falsos     = {"FALSO", "FALSE", "0", "NO", "N"}

    out = s.isin(verdaderos)
    # lo que no cae en verdaderos queda False (incluye falsos y basura)
    return out.fillna(False)


def construir_metricas_radar_para_config(
    bloque: pd.DataFrame,
    resumen_sg_sv: pd.DataFrame,
    total_sg: int,
) -> dict:
    """
    Métricas para radar (0..1). Alineadas a PC-SMOTE.

    Ejes:
      - tasa_pasa_pureza
      - tasa_pasa_densidad
      - tasa_pasa_riesgo
      - tasa_semillas_candidatas
      - equidad_generacion
      - cobertura_clases   (clases con SG>0 / total clases)
    Además (para texto/diagnóstico):
      - total_clases
      - clases_con_sg
      - sg_total
    """

    def tasa_pasa(col: str) -> float:
        if col not in bloque.columns or len(bloque) == 0:
            return np.nan
        mask = _normalizar_flag_a_bool(bloque[col])
        return float(mask.mean())

    tasa_pureza = tasa_pasa("pasa_pureza")
    tasa_densidad = tasa_pasa("pasa_densidad")
    tasa_riesgo = tasa_pasa("pasa_riesgo")

    tasa_candidatas = np.nan
    if len(bloque) > 0 and all(c in bloque.columns for c in ["pasa_pureza", "pasa_densidad", "pasa_riesgo"]):
        m_p = _normalizar_flag_a_bool(bloque["pasa_pureza"])
        m_d = _normalizar_flag_a_bool(bloque["pasa_densidad"])
        m_r = _normalizar_flag_a_bool(bloque["pasa_riesgo"])
        tasa_candidatas = float((m_p & m_d & m_r).mean())

    # Cobertura y K
    cobertura = np.nan
    total_clases = np.nan
    clases_con_sg = np.nan
    if resumen_sg_sv is not None and len(resumen_sg_sv) > 0 and "SG" in resumen_sg_sv.columns:
        clases_con_sg = int((resumen_sg_sv["SG"] > 0).sum())
        total_clases = int(len(resumen_sg_sv))
        cobertura = float(clases_con_sg / total_clases) if total_clases > 0 else np.nan

    equidad_generacion = _equidad_generacion_desde_bloque(bloque)

    return {
        "tasa_pasa_pureza": tasa_pureza,
        "tasa_pasa_densidad": tasa_densidad,
        "tasa_pasa_riesgo": tasa_riesgo,
        "tasa_semillas_candidatas": tasa_candidatas,
        "equidad_generacion": equidad_generacion,
        "cobertura_clases": cobertura,     # <-- VOLVIÓ
        "total_clases": total_clases,      # <-- para mostrar en gráfico
        "clases_con_sg": clases_con_sg,    # <-- para mostrar en gráfico
        "sg_total": float(total_sg),       # por si lo querés
    }





def graficar_radar_comparacion_configuraciones(
    dataset: str,
    metricas_por_config: dict,
    configuraciones_a_comparar: list,
    directorio_graficos: str = "graficos",
    guardar: bool = True,
    mostrar_en_notebook: bool = True,
    dpi: int = 140,
) -> None:
    """
    Dibuja un radar superponiendo 2..5 configuraciones de un mismo dataset.
    Normaliza sg_total a 0..1 dentro del dataset (solo entre las configs comparadas).
    """

    # Validación mínima
    if not configuraciones_a_comparar:
        print(f"  ▶ Radar: sin configuraciones a comparar para {dataset}")
        return

    # Armar filas en orden
    filas = []
    for cfg in configuraciones_a_comparar:
        if cfg not in metricas_por_config:
            continue
        d = metricas_por_config[cfg].copy()
        d["configuracion"] = cfg
        filas.append(d)

    if len(filas) < 2:
        print(f"  ▶ Radar: datos insuficientes para comparar en {dataset} (n<2)")
        return

    df = pd.DataFrame(filas)


    # K (clases totales) — idealmente es igual para todas las configs
    k = None
    if "total_clases" in df.columns:
        valores_k = df["total_clases"].dropna().unique().tolist()
        if len(valores_k) >= 1:
            k = int(valores_k[0])

    # Normalizar sg_total dentro de las configs comparadas (min-max)
    if "sg_total" in df.columns:
        sg_min = float(df["sg_total"].min())
        sg_max = float(df["sg_total"].max())
        if sg_max > sg_min:
            df["sg_total_norm"] = (df["sg_total"] - sg_min) / (sg_max - sg_min)
        else:
            df["sg_total_norm"] = 0.0
    else:
        df["sg_total_norm"] = np.nan

    # Ejes del radar (todos 0..1, y "más alto = mejor")
    ejes = [
        ("Pasa pureza", "tasa_pasa_pureza"),
        ("Pasa densidad", "tasa_pasa_densidad"),
        ("Pasa riesgo", "tasa_pasa_riesgo"),
        ("Semillas candidatas validas", "tasa_semillas_candidatas"),
        ("Reparto de Semillas", "equidad_generacion"),
        ("Clases Alcanzadas", "cobertura_clases"),
    ]


    etiquetas = [e[0] for e in ejes]
    cols = [e[1] for e in ejes]

    # Ángulos
    n = len(ejes)
    angulos = np.linspace(0, 2 * np.pi, n, endpoint=False).tolist()
    angulos += angulos[:1]  # cerrar polígono

    fig = plt.figure(figsize=(7.2, 6.2))
    ax = plt.subplot(111, polar=True)

    # Dibujar cada config
    for _, row in df.iterrows():
        valores = []
        for c in cols:
            v = row.get(c, np.nan)
            # si algo falta, lo llevamos a 0 para que sea visible el “fracaso”
            if pd.isna(v):
                v = 0.0
            # clamp 0..1
            v = float(max(0.0, min(1.0, v)))
            valores.append(v)

        valores += valores[:1]  # cerrar polígono

        # Sin fijar colores (usa el ciclo por defecto de matplotlib)
        ax.plot(angulos, valores, linewidth=2, label=row["configuracion"])
        ax.fill(angulos, valores, alpha=0.08)

    # Estética base
    ax.set_xticks(angulos[:-1])
    ax.set_xticklabels(etiquetas, fontsize=10)
    ax.set_ylim(0, 1)
    if k is not None:
        ax.set_title(f"{dataset} | Radar comparativo (configs)  —  C={k} clases", y=1.08)
    else:
        ax.set_title(f"{dataset} | Radar comparativo (configs)", y=1.08)


    # Leyenda afuera
    ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), frameon=False, fontsize=9)

    # # -------------------------------------------------
    # # Recuadro informativo: cobertura por configuración
    # # -------------------------------------------------
    # if all(c in df.columns for c in ["clases_con_sg", "total_clases"]):
    #     lineas = ["Cobertura generativa (clases):"]
    #     for _, row in df.iterrows():
    #         cfg = row["configuracion"]
    #         a = row["clases_con_sg"]
    #         b = row["total_clases"]
    #         if pd.notna(a) and pd.notna(b) and float(b) > 0:
    #             lineas.append(f"• {cfg}: {int(a)}/{int(b)}")

    #     fig.text(
    #         1.02, 0.55,                # posición a la derecha
    #         "\n".join(lineas),
    #         ha="left",
    #         va="top",
    #         fontsize=9
    #     )

    #     # achica el área del radar para que no se pise
    #     plt.tight_layout(rect=[0, 0, 0.80, 1])
    # else:
    #     plt.tight_layout()


    if guardar:
        out_dir = Path(directorio_graficos)
        out_dir.mkdir(parents=True, exist_ok=True)
        nombre_archivo = f"{dataset}__radar_configs.png".replace(" ", "_")
        fig.savefig(out_dir / nombre_archivo, dpi=dpi, bbox_inches="tight")

    if mostrar_en_notebook:
        plt.show()

    plt.close(fig)


In [15]:
import os
import glob
import pandas as pd

# =========================
# DATASETS A GRAFICAR
# =========================
datasets_a_graficar = {
    "predict_faults",
    "ecoli",
    "heart",
    "glass",
    "wdbc",
    "telco_churn",
    "us_crime",
    "shuttle",
}

# =========================
# CONFIGURACIÓN "DESTACADA" (para SG/SV + DotBox)
# =========================
configuraciones_a_graficar = {
    "predict_faults": "PRD85_PR50_CPprop_UD050_UR055_Upp045_I0",
    "heart":         "PRD85_PR50_CPprop_UD050_UR055_Upp045_I0",
    "glass":         "PRD85_PR30_CPprop_UD060_UR040_Upp060_I0",
    "ecoli":         "PRD70_PR40_CPent_UD060_UR045_PE70_I0",
    "telco_churn":   "PRD85_PR40_CPent_UD060_UR045_PE70_I0",
    "us_crime":      "PRD85_PR40_CPent_UD045_UR045_PE80_I0",
    "wdbc":          "PRD70_PR30_CPprop_UD050_UR040_Upp060_I0",
    "shuttle":       "PRD70_PR30_CPprop_UD025_UR030_Upp040_I1",
}

# =========================
# CONFIGURACIONES A COMPARAR (TOP-K POR DATASET) -> RADAR
# =========================
configuraciones_a_comparar = {
    "predict_faults": [
        "PRD85_PR50_CPprop_UD050_UR055_Upp045_I0",
        "PRD85_PR50_CPprop_UD050_UR055_Upp045_I1",
        "PRD85_PR40_CPent_UD050_UR045_PE60_I0",
        "PRD85_PR40_CPent_UD050_UR045_PE60_I1",
    ],
    "telco_churn": [
        "PRD85_PR40_CPent_UD060_UR045_PE70_I0",
        "PRD85_PR40_CPent_UD050_UR045_PE70_I0",
        "PRD85_PR30_CPprop_UD060_UR040_Upp060_I1",
        "PRD70_PR30_CPprop_UD060_UR040_Upp060_I3",
    ],
    "ecoli" : [
        "PRD70_PR40_CPent_UD060_UR045_PE70_I0",
        "PRD70_PR30_CPprop_UD060_UR040_Upp060_I0",
        "PRD70_PR30_CPprop_UD050_UR040_Upp060_I0",
    ],
    "us_crime"  :[
        "PRD85_PR40_CPent_UD035_UR045_PE60_I1",
        "PRD85_PR40_CPent_UD045_UR045_PE80_I0",
        "PRD85_PR40_CPent_UD035_UR045_PE80_I0",
        "PRD90_PR40_CPent_UD035_UR045_PE60_I1",
    ],
    "wdbc" :[
        "PRD70_PR30_CPprop_UD050_UR040_Upp060_I0",
        "PRD70_PR30_CPprop_UD060_UR040_Upp060_I0",
        "PRD85_PR40_CPent_UD050_UR045_PE60_I0",
        "PRD85_PR40_CPent_UD050_UR045_PE70_I0",
    ],
    "heart" : [
        "PRD70_PR30_CPprop_UD050_UR040_Upp060_I3",
        "PRD85_PR50_CPprop_UD050_UR055_Upp045_I0",
        "PRD85_PR50_CPprop_UD050_UR055_Upp045_I3",
        "PRD70_PR30_CPprop_UD050_UR040_Upp060_I0",
    ],
    "glass" :[
        "PRD70_PR30_CPprop_UD050_UR040_Upp060_I1",
        "PRD85_PR30_CPprop_UD050_UR040_Upp060_I1",
        "PRD70_PR30_CPprop_UD050_UR040_Upp060_I0",
        "PRD85_PR30_CPprop_UD050_UR040_Upp060_I0",
    ],
    "shuttle" : [
        "PRD70_PR30_CPprop_UD025_UR030_Upp040_I1",
        "PRD70_PR30_CPprop_UD050_UR040_Upp060_I0",
        "PRD70_PR30_CPprop_UD040_UR035_Upp050_I1",
        "PRD70_PR30_CPprop_UD035_UR030_Upp045_I1",
    ]
}

# =========================
# CARGA DISTRIBUCIONES TEST
# =========================
def cargar_distribuciones_test(ruta_test: str) -> dict:
    archivos_test = glob.glob(os.path.join(ruta_test, "*_test.csv"))
    dist_test = {}

    for ruta in archivos_test:
        nombre = os.path.basename(ruta)
        dataset = nombre.split("_tdataset")[0]

        df_test = pd.read_csv(ruta)
        col_clase = df_test.columns[-1]
        dist_test[dataset] = df_test[col_clase].value_counts()

    return dist_test


# =========================
# ANÁLISIS 1: SG/SV POR CLASE
# =========================
def construir_resumen_sg_sv_por_clase(bloque: pd.DataFrame) -> pd.DataFrame:
    resumen = (
        bloque
        .groupby("clase_objetivo")
        .agg(
            SG=("synthetics_from_this_seed", "sum"),
            SV=("es_semilla_valida", "sum"),
        )
        .sort_index()
    )
    return resumen


def imprimir_resumen_sg_sv(resumen: pd.DataFrame) -> int:
    for clase, fila in resumen.iterrows():
        sg = int(fila["SG"])
        sv = int(fila["SV"])
        ratio = sg / sv if sv > 0 else 0
        print(f"{clase:<6} SV={sv:<3} SG={sg:<4} SG/SV={ratio:.2f}")

    total_sg = int(resumen["SG"].sum())
    cobertura = int((resumen["SG"] > 0).sum())
    total_clases = int(len(resumen))

    print(f"\n  ▶ Total sintéticos (SG): {total_sg}")
    print(f"  ▶ Cobertura de clases  : {cobertura}/{total_clases}")

    if total_sg == 0:
        print("  ⚠ Configuración inválida (SG = 0)")

    return total_sg


# =========================
# ANÁLISIS 2: DESALINEACIÓN VS TEST
# =========================
def imprimir_desalineacion_vs_test(dataset: str, resumen: pd.DataFrame, dist_test: dict) -> None:
    if dataset not in dist_test:
        return

    print("  ▶ Desalineación SG vs TEST:")
    for clase in resumen.index:
        sg = int(resumen.loc[clase, "SG"])
        ft = int(dist_test[dataset].get(clase, 0))
        delta = sg - ft
        print(f"     {clase:<6} SG={sg:<4} TEST={ft:<3} Δ={delta}")


# =========================
# ANÁLISIS 3: ENTROPÍA (media/std/CV) EN SV=1
# =========================
def analizar_entropia_en_semillas_validas(bloque: pd.DataFrame) -> None:
    # Si el criterio de pureza es "proporcion", entropía NO aplica
    if "criterio_pureza" in bloque.columns:
        criterio = str(bloque["criterio_pureza"].iloc[0]).strip().lower()
        if "prop" in criterio:
            print("  ▶ Entropía: no aplica (criterio_pureza=proporcion)")
            return

    if not tiene_columnas(bloque, ["entropia", "es_semilla_valida"]):
        print("  ▶ Entropía: (omitido, faltan columnas)")
        return

    semillas_validas = bloque[bloque["es_semilla_valida"] == 1]
    if len(semillas_validas) == 0:
        print("  ▶ Entropía: sin semillas válidas")
        return

    ent = convertir_a_float_seguro(semillas_validas["entropia"]).dropna()
    if len(ent) == 0:
        print("  ▶ Entropía: todos NaN tras conversión")
        return

    media = float(ent.mean())
    std = float(ent.std(ddof=1)) if len(ent) > 1 else 0.0
    cv = (std / media) if media != 0 else 0.0

    print("  ▶ Entropía (solo SV=1):")
    print(f"     n={len(ent)} | media={media:.4f} | std={std:.4f} | CV={cv:.4f}")


# =========================
# ANÁLISIS 4: CORRELACIÓN entropía ↔ SG_por_semilla (Spearman) EN SV=1
# =========================
def analizar_correlacion_entropia_vs_sg(bloque: pd.DataFrame) -> None:
    # Si el criterio de pureza es "proporcion", entropía NO aplica
    if "criterio_pureza" in bloque.columns:
        criterio = str(bloque["criterio_pureza"].iloc[0]).strip().lower()
        if "prop" in criterio:
            print("  ▶ Correlación entropía↔SG: no aplica (criterio_pureza=proporcion)")
            return

    cols = ["entropia", "synthetics_from_this_seed", "es_semilla_valida"]
    if not tiene_columnas(bloque, cols):
        print("  ▶ Correlación entropía↔SG: (omitido, faltan columnas)")
        return

    semillas_validas = bloque[bloque["es_semilla_valida"] == 1].copy()
    if len(semillas_validas) < 3:
        print("  ▶ Correlación entropía↔SG: datos insuficientes (n<3)")
        return

    x = convertir_a_float_seguro(semillas_validas["entropia"])
    y = convertir_a_float_seguro(semillas_validas["synthetics_from_this_seed"])

    tmp = pd.DataFrame({"x": x, "y": y}).dropna()
    if len(tmp) < 3:
        print("  ▶ Correlación entropía↔SG: datos insuficientes tras limpieza")
        return

    corr = tmp["x"].corr(tmp["y"], method="spearman")
    if pd.isna(corr):
        print("  ▶ Correlación entropía↔SG: no calculable")
        return

    print("  ▶ Correlación (Spearman) entropía ↔ SG_por_semilla (solo SV=1):")
    print(f"     rho={float(corr):.4f}")


# =========================
# ANÁLISIS 5: EFICIENCIA DE SEMILLAS (SV activas)
# =========================
def analizar_eficiencia_de_semillas(bloque: pd.DataFrame) -> None:
    cols = ["es_semilla_valida", "synthetics_from_this_seed"]
    if not tiene_columnas(bloque, cols):
        print("  ▶ Eficiencia semillas: (omitido, faltan columnas)")
        return

    semillas_validas = bloque[bloque["es_semilla_valida"] == 1].copy()
    if len(semillas_validas) == 0:
        print("  ▶ Eficiencia semillas: sin semillas válidas")
        return

    total_sv = int(len(semillas_validas))

    sg_por_semilla = convertir_a_float_seguro(semillas_validas["synthetics_from_this_seed"])
    semillas_validas["sg_float"] = sg_por_semilla

    activas = semillas_validas[semillas_validas["sg_float"] > 0]
    cant_activas = int(len(activas))
    pct_activas = (cant_activas / total_sv) if total_sv > 0 else 0.0

    if cant_activas > 0:
        media_sg_activas = float(activas["sg_float"].mean())
        mediana_sg_activas = float(activas["sg_float"].median())
    else:
        media_sg_activas = 0.0
        mediana_sg_activas = 0.0

    print("  ▶ Eficiencia de semillas (solo SV=1):")
    print(f"     SV_totales={total_sv} | SV_activas={cant_activas} ({pct_activas*100:.2f}%)")
    print(f"     SG_por_semilla_activa: media={media_sg_activas:.3f} | mediana={mediana_sg_activas:.3f}")


# =========================
# ANÁLISIS 6: SV=1 vs SV=0 (entropia/densidad/riesgo)
# =========================
def analizar_diferencias_sv_vs_no_sv(bloque: pd.DataFrame) -> None:
    if "es_semilla_valida" not in bloque.columns:
        print("  ▶ Comparación SV vs no-SV: (omitido, falta es_semilla_valida)")
        return

    columnas = []
    for c in ["entropia", "densidad", "riesgo"]:
        if c in bloque.columns:
            columnas.append(c)

    if len(columnas) == 0:
        print("  ▶ Comparación SV vs no-SV: (omitido, faltan entropia/densidad/riesgo)")
        return

    sv = bloque[bloque["es_semilla_valida"] == 1]
    no_sv = bloque[bloque["es_semilla_valida"] == 0]

    if len(sv) == 0 or len(no_sv) == 0:
        print("  ▶ Comparación SV vs no-SV: datos insuficientes (uno de los grupos vacío)")
        return

    print("  ▶ Diferencias SV=1 vs SV=0 (media y std):")
    for c in columnas:
        v1 = convertir_a_float_seguro(sv[c]).dropna()
        v0 = convertir_a_float_seguro(no_sv[c]).dropna()

        if len(v1) == 0 or len(v0) == 0:
            print(f"     {c:<8} (omitido: NaN tras conversión)")
            continue

        media_1 = float(v1.mean())
        std_1 = float(v1.std(ddof=1)) if len(v1) > 1 else 0.0

        media_0 = float(v0.mean())
        std_0 = float(v0.std(ddof=1)) if len(v0) > 1 else 0.0

        print(f"     {c:<8} SV=1 media={media_1:.4f} std={std_1:.4f} | SV=0 media={media_0:.4f} std={std_0:.4f}")


# =========================
# ANÁLISIS 7: FRONTERA (pasa_* vs no pasa_*) con flags VERDADERO/FALSO
# =========================
def analizar_frontera_por_criterio(bloque: pd.DataFrame) -> None:
    columnas_base = ["pasa_pureza", "pasa_densidad", "pasa_riesgo", "densidad", "riesgo"]
    for col in columnas_base:
        if col not in bloque.columns:
            print("  ▶ Frontera por criterio: (omitido, faltan columnas)")
            return

    # Elegir variable para pureza según criterio_pureza
    col_valor_pureza = "entropia"
    if "criterio_pureza" in bloque.columns:
        criterio = str(bloque["criterio_pureza"].iloc[0]).strip().lower()
        if "prop" in criterio:
            col_valor_pureza = "proporcion_min_valor"
        else:
            col_valor_pureza = "entropia"

    if col_valor_pureza not in bloque.columns:
        print(f"  ▶ Frontera por criterio: (omitido, falta {col_valor_pureza})")
        return

    criterios = [
        {"nombre": "pureza",   "col_pasa": "pasa_pureza",   "col_valor": col_valor_pureza},
        {"nombre": "densidad", "col_pasa": "pasa_densidad", "col_valor": "densidad"},
        {"nombre": "riesgo",   "col_pasa": "pasa_riesgo",   "col_valor": "riesgo"},
    ]

    print("  ▶ Análisis de frontera por criterio:")

    for crit in criterios:
        nombre = crit["nombre"]
        col_pasa = crit["col_pasa"]
        col_valor = crit["col_valor"]

        flag = bloque[col_pasa].astype(str).str.strip().str.upper()
        mask_pasa = flag.eq("VERDADERO")
        mask_no = flag.eq("FALSO")

        pasa = bloque.loc[mask_pasa, col_valor]
        no_pasa = bloque.loc[mask_no, col_valor]

        pasa_num = convertir_a_float_seguro(pasa).dropna()
        no_num = convertir_a_float_seguro(no_pasa).dropna()

        if len(pasa_num) == 0 or len(no_num) == 0:
            print(f"     {nombre:<8}: datos insuficientes (NaN tras conversión)")
            continue

        media_pasa = float(pasa_num.mean())
        std_pasa = float(pasa_num.std(ddof=1)) if len(pasa_num) > 1 else 0.0

        media_no = float(no_num.mean())
        std_no = float(no_num.std(ddof=1)) if len(no_num) > 1 else 0.0

        delta_media = media_pasa - media_no

        print(
            f"     {nombre:<8} [{col_valor}] | "
            f"pasa: n={len(pasa_num):<4} μ={media_pasa:.4f} σ={std_pasa:.4f} | "
            f"no pasa: n={len(no_num):<4} μ={media_no:.4f} σ={std_no:.4f} | "
            f"Δμ={delta_media:.4f}"
        )


# =========================
# EJECUCIÓN PRINCIPAL
# =========================
RUTA_LOGS = "."
RUTA_TEST = "."

archivos_xlsx = [
    f for f in glob.glob(os.path.join(RUTA_LOGS, "*.xlsx"))
    if not os.path.basename(f).startswith("~$")
]

dist_test = cargar_distribuciones_test(RUTA_TEST)

for ruta in archivos_xlsx:
    nombre_archivo = os.path.basename(ruta)
    dataset = nombre_archivo.replace("log_pcsmote_x_muestra_", "").replace(".xlsx", "")

    df = pd.read_excel(ruta, engine="openpyxl")
    validar_columnas_obligatorias(df, nombre_archivo)

    if dataset not in datasets_a_graficar:
        continue

    print("\n" + "=" * 100)
    print(f"Dataset: {dataset}")
    print("=" * 100)

    # acumulador radar (por dataset)
    metricas_radar_por_config = {}

    for configuracion, bloque in df.groupby("configuracion"):
        print(f"\n{configuracion}")
        print("-" * len(configuracion))

        # 1) Resumen SG/SV por clase
        resumen = construir_resumen_sg_sv_por_clase(bloque)
        total_sg = imprimir_resumen_sg_sv(resumen)

        # guardar métricas para radar SIEMPRE
        metricas_radar_por_config[configuracion] = construir_metricas_radar_para_config(
            bloque=bloque,
            resumen_sg_sv=resumen,
            total_sg=total_sg,
        )

        # 2) Desalineación vs TEST
        if total_sg > 0:
            imprimir_desalineacion_vs_test(dataset, resumen, dist_test)

            # gráfico SG/SV solo para la config destacada
            if dataset in configuraciones_a_graficar and configuracion == configuraciones_a_graficar[dataset]:
                graficar_sg_sv_por_clase(
                    resumen=resumen,
                    dataset=dataset,
                    configuracion=configuracion,
                    directorio_graficos="graficos",
                    mostrar_en_notebook=False,
                    guardar=True
                )

        # 3) Entropía (dispersión) en semillas válidas
        analizar_entropia_en_semillas_validas(bloque)

        # 4) Correlación entropía vs SG_por_semilla
        analizar_correlacion_entropia_vs_sg(bloque)

        # 5) Eficiencia de semillas
        analizar_eficiencia_de_semillas(bloque)

        # 6) SV=1 vs SV=0
        analizar_diferencias_sv_vs_no_sv(bloque)

        # 7) Frontera por criterio (pasa_* vs no pasa_*)
        analizar_frontera_por_criterio(bloque)

        # 7b) Dot + Box por criterio (solo para config destacada)
        if dataset in configuraciones_a_graficar and configuracion == configuraciones_a_graficar[dataset]:
            graficar_dot_box_por_criterio(
                bloque=bloque,
                dataset=dataset,
                configuracion=configuracion,
                directorio_graficos="graficos",
                guardar=True,
                mostrar_en_notebook=False,
                dpi=140,
                col_clase="clase_objetivo"
            )

    # =========================
    # RADAR (TOP-K POR DATASET) -> 1 vez por dataset (FUERA del loop)
    # =========================
    configs_cmp = configuraciones_a_comparar.get(dataset, [])
    configs_cmp = [c for c in configs_cmp if c in metricas_radar_por_config]

    if len(configs_cmp) >= 2:
        graficar_radar_comparacion_configuraciones(
            dataset=dataset,
            metricas_por_config=metricas_radar_por_config,
            configuraciones_a_comparar=configs_cmp,
            directorio_graficos="graficos",
            guardar=True,
            mostrar_en_notebook=False,
            dpi=140
        )



Dataset: ecoli

PRD70_PR30_CPprop_UD050_UR040_Upp060_I0
---------------------------------------
im     SV=27  SG=53   SG/SV=1.96
imU    SV=6   SG=86   SG/SV=14.33
om     SV=7   SG=98   SG/SV=14.00
pp     SV=23  SG=73   SG/SV=3.17

  ▶ Total sintéticos (SG): 310
  ▶ Cobertura de clases  : 4/4
  ▶ Entropía: no aplica (criterio_pureza=proporcion)
  ▶ Correlación entropía↔SG: no aplica (criterio_pureza=proporcion)
  ▶ Eficiencia de semillas (solo SV=1):
     SV_totales=63 | SV_activas=56 (88.89%)
     SG_por_semilla_activa: media=5.536 | mediana=3.000
  ▶ Diferencias SV=1 vs SV=0 (media y std):
     entropia (omitido: NaN tras conversión)
     densidad SV=1 media=0.8957 std=0.1627 | SV=0 media=0.5542 std=0.4074
     riesgo   SV=1 media=0.0544 std=0.0905 | SV=0 media=0.1704 std=0.2887
  ▶ Análisis de frontera por criterio:
     pureza  : datos insuficientes (NaN tras conversión)
     densidad: datos insuficientes (NaN tras conversión)
     riesgo  : datos insuficientes (NaN tras conversión

c:\Users\FamiliaNatelloMedina\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\core\nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
c:\Users\FamiliaNatelloMedina\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\core\nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


  ▶ Eficiencia de semillas (solo SV=1):
     SV_totales=107 | SV_activas=70 (65.42%)
     SG_por_semilla_activa: media=1.643 | mediana=1.000
  ▶ Diferencias SV=1 vs SV=0 (media y std):
     entropia SV=1 media=0.0000 std=0.0000 | SV=0 media=0.4961 std=0.4029
     densidad SV=1 media=0.9773 std=0.0881 | SV=0 media=0.6327 std=0.4566
     riesgo   SV=1 media=0.0000 std=0.0000 | SV=0 media=0.1633 std=0.2525
  ▶ Análisis de frontera por criterio:
     pureza  : datos insuficientes (NaN tras conversión)
     densidad: datos insuficientes (NaN tras conversión)
     riesgo  : datos insuficientes (NaN tras conversión)

PRD85_PR40_CPent_UD050_UR045_PE70_I0
------------------------------------
M      SV=107 SG=115  SG/SV=1.07

  ▶ Total sintéticos (SG): 115
  ▶ Cobertura de clases  : 1/1
  ▶ Entropía (solo SV=1):
     n=107 | media=0.0000 | std=0.0000 | CV=0.0000
  ▶ Correlación entropía↔SG: no calculable
  ▶ Eficiencia de semillas (solo SV=1):
     SV_totales=107 | SV_activas=70 (65.42%)
     SG